# Experiment 02: Physical GPU VRAM Profiling Across All MLP Submodules

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Objective:
Measure and verify **physical GPU VRAM reduction** by structurally replacing dense `nn.Linear` layers with custom parameter-efficient factorized modules (`TuckerFactorizedRowLinear` and `TuckerFactorizedColLinear`):
1. **Uncompressed Baseline**: Standard dense matrices ($6912 \times 1152$ and $1152 \times 6912$) across all 78 MLP projections ($621.08\text{M}$ parameters / $2,484.34\text{ MB}$ in FP32).
2. **All-MLP DBSCAN Tucker Processed**:
   Stores ONLY Tucker core $\mathcal{S} \in \mathbb{R}^{3 \times 100 \times 350}$, factor matrices, and uncompressed superweights on the GPU.
   Eliminates $\mathbf{172,893,396\text{ parameters}}$ (**$\approx 691.57\text{ MB}$ physical GPU memory reduction** in FP32).

### Metrics Profiled:
- **Static Parameter VRAM (MB)**: Actual CUDA memory allocated for model weights.
- **Allocated GPU Memory (MB)**: Active memory allocated by PyTorch allocator (`torch.cuda.memory_allocated()`).
- **Peak Runtime VRAM (MB)**: Maximum peak memory reached during live MNLI inference (`torch.cuda.max_memory_allocated()`).
- **Downstream Accuracy Retention**: Evaluated on GLUE MNLI validation set.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
import gc
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded neural_decomp library.
Device: NVIDIA GeForce RTX 3070 Ti | CUDA Available: True


In [2]:
# =====================================================================
# STEP 2: Precise GPU VRAM Measurement Utilities
# =====================================================================
def get_model_param_vram_mb(mod):
    """Calculates exact memory (MB) occupied by model parameters and buffers on CUDA."""
    param_bytes = sum(p.numel() * p.element_size() for p in mod.parameters() if p.is_cuda)
    buffer_bytes = sum(b.numel() * b.element_size() for b in mod.buffers() if b.is_cuda)
    return (param_bytes + buffer_bytes) / (1024 ** 2)

def get_cuda_memory_snapshot():
    """Returns current allocated, reserved, and peak allocated VRAM in MB."""
    if not torch.cuda.is_available():
        return {"allocated_mb": 0.0, "reserved_mb": 0.0, "max_allocated_mb": 0.0}
    return {
        "allocated_mb": round(torch.cuda.memory_allocated() / (1024 ** 2), 2),
        "reserved_mb": round(torch.cuda.memory_reserved() / (1024 ** 2), 2),
        "max_allocated_mb": round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2),
    }

def clear_cuda_cache():
    """Clears unused cached memory from the PyTorch allocator and resets peak stats."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

print("VRAM profiling utilities ready.")

VRAM profiling utilities ready.
time: 0.00s
cummulative_time: 1.72s


In [3]:
# =====================================================================
# STEP 3: Initialize Model & Tokenizer on GPU (Unprocessed Baseline)
# =====================================================================
clear_cuda_cache()
model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
D_IN = model.model.layers[0].mlp.gate_proj.weight.shape[1]
D_OUT = model.model.layers[0].mlp.gate_proj.weight.shape[0]

unprocessed_param_vram = get_model_param_vram_mb(model)
print(f"Loaded {model_id} on GPU ({NUM_LAYERS} Decoder Layers)")
print(f"Static Model Parameter VRAM: {unprocessed_param_vram:.2f} MB")

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 369.77it/s]


Loaded google/gemma-3-1b-it on GPU (26 Decoder Layers)
Static Model Parameter VRAM: 3814.26 MB
time: 4.53s
cummulative_time: 6.25s


In [4]:
# =====================================================================
# STEP 4: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

EVAL_SAMPLE_COUNT = 1000
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

Loaded GLUE MNLI: 9,815 total samples | Active Evaluation Subset: 1,000 samples
time: 3.56s
cummulative_time: 9.81s


## Step 1: Unprocessed Model Inference & Peak VRAM Profiling

We measure the baseline model's static memory, allocated memory, and peak runtime memory during GLUE MNLI inference passes while capturing intermediate neuron activation profiles.

In [5]:
# =====================================================================
# STEP 5: Baseline Inference & Peak VRAM Profiling
# =====================================================================
clear_cuda_cache()

layer_trajectories = {l: [] for l in range(NUM_LAYERS)}
current_acts = {}

def make_act_hook(layer_idx):
    def hook_fn(module, input_tensor, output_tensor):
        act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        current_acts[layer_idx] = act.detach().cpu()
    return hook_fn

hooks = [
    model.model.layers[l].mlp.act_fn.register_forward_hook(make_act_hook(l))
    for l in range(NUM_LAYERS)
]

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Unprocessed Model Inference & Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        for l in range(NUM_LAYERS):
            if l in current_acts and current_acts[l] is not None:
                pooled = current_acts[l].squeeze(0).mean(dim=0).numpy()
                layer_trajectories[l].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in hooks:
    h.remove()

baseline_snapshot = get_cuda_memory_snapshot()
acts_matrices = {l: np.stack(layer_trajectories[l]) for l in range(NUM_LAYERS)}
baseline_accuracy = accuracy_score(ground_truth, predictions)
baseline_peak_vram = baseline_snapshot["max_allocated_mb"]

print(f"\nUnprocessed Model Baseline Profile:")
print(f"  Downstream Accuracy:     {baseline_accuracy * 100:.2f}%")
print(f"  Static Parameter VRAM:   {unprocessed_param_vram:.2f} MB")
print(f"  Allocated GPU VRAM:      {baseline_snapshot['allocated_mb']:.2f} MB")
print(f"  Peak Runtime VRAM:       {baseline_peak_vram:.2f} MB")

Unprocessed Model Inference & Profiling: 100%|██████████| 1000/1000 [00:45<00:00, 21.93it/s]



Unprocessed Model Baseline Profile:
  Downstream Accuracy:     48.00%
  Static Parameter VRAM:   3814.26 MB
  Allocated GPU VRAM:      3942.44 MB
  Peak Runtime VRAM:       4128.81 MB
time: 46.02s
cummulative_time: 55.84s


## Step 2: Custom Factorized Modules for Physical GPU Memory Reduction

To physically eliminate GPU memory, we replace standard dense `nn.Linear` layers with:
1. **`TuckerFactorizedRowLinear`**:
   For row-partitioned projections (`gate_proj`, `up_proj`). Stores uncompressed superweights, uncompressed remaining coordinates, and the compact Tucker core/factors $\mathcal{S}, U^{(1)}, U^{(2)}, U^{(3)}$.
2. **`TuckerFactorizedColLinear`**:
   For column-partitioned projections (`down_proj`). Evaluates $x W^T$ by linearly splitting into superweight, remaining, and factorized active products without ever allocating the full matrix.

In [6]:
# =====================================================================
# STEP 6: Define TuckerFactorizedRowLinear and TuckerFactorizedColLinear
# =====================================================================
class TuckerFactorizedRowLinear(torch.nn.Module):
    """
    Physically factorized linear module for row-sliced projections (gate_proj, up_proj).
    Stores only Tucker core, factor matrices, superweights, and remaining weights on GPU.
    """
    def __init__(self, W_orig, super_indices, active_coords, core, factors):
        super().__init__()
        self.out_features, self.in_features = W_orig.shape
        self.register_buffer("super_indices", torch.tensor(super_indices, dtype=torch.long))
        self.register_buffer("active_coords", torch.tensor(active_coords, dtype=torch.long))
        all_special = set(super_indices).union(set(active_coords))
        remaining_indices = [i for i in range(self.out_features) if i not in all_special]
        self.register_buffer("remaining_indices", torch.tensor(remaining_indices, dtype=torch.long))

        self.superweights = torch.nn.Parameter(W_orig[super_indices, :].clone(), requires_grad=False)
        self.remaining_weights = torch.nn.Parameter(W_orig[remaining_indices, :].clone(), requires_grad=False)
        self.core = torch.nn.Parameter(core.clone(), requires_grad=False)
        self.factors = torch.nn.ParameterList([
            torch.nn.Parameter(f.clone(), requires_grad=False) for f in factors
        ])

    def forward(self, x):
        orig_shape = x.shape
        x_2d = x.reshape(-1, self.in_features)
        out = torch.empty((x_2d.shape[0], self.out_features), device=x.device, dtype=x.dtype)
        out[:, self.super_indices] = (x_2d @ self.superweights.T).to(dtype=out.dtype)
        out[:, self.remaining_indices] = (x_2d @ self.remaining_weights.T).to(dtype=out.dtype)
        W_active = tucker_to_tensor((self.core, list(self.factors))).reshape(-1, self.in_features)
        out[:, self.active_coords] = (x_2d @ W_active.T).to(dtype=out.dtype)
        return out.reshape(*orig_shape[:-1], self.out_features)


class TuckerFactorizedColLinear(torch.nn.Module):
    """
    Physically factorized linear module for column-sliced projections (down_proj).
    Stores only Tucker core, factor matrices, superweights, and remaining weights on GPU.
    """
    def __init__(self, W_orig, super_indices, active_coords, core, factors):
        super().__init__()
        self.out_features, self.in_features = W_orig.shape
        self.register_buffer("super_indices", torch.tensor(super_indices, dtype=torch.long))
        self.register_buffer("active_coords", torch.tensor(active_coords, dtype=torch.long))
        all_special = set(super_indices).union(set(active_coords))
        remaining_indices = [i for i in range(self.in_features) if i not in all_special]
        self.register_buffer("remaining_indices", torch.tensor(remaining_indices, dtype=torch.long))

        self.superweights = torch.nn.Parameter(W_orig[:, super_indices].clone(), requires_grad=False)
        self.remaining_weights = torch.nn.Parameter(W_orig[:, remaining_indices].clone(), requires_grad=False)
        self.core = torch.nn.Parameter(core.clone(), requires_grad=False)
        self.factors = torch.nn.ParameterList([
            torch.nn.Parameter(f.clone(), requires_grad=False) for f in factors
        ])

    def forward(self, x):
        orig_shape = x.shape
        x_2d = x.reshape(-1, self.in_features)
        W_active = tucker_to_tensor((self.core, list(self.factors))).reshape(-1, self.out_features)
        out = (
            x_2d[:, self.super_indices] @ self.superweights.T +
            x_2d[:, self.remaining_indices] @ self.remaining_weights.T +
            x_2d[:, self.active_coords] @ W_active
        )
        return out.reshape(*orig_shape[:-1], self.out_features)

print("Custom Tucker Factorized Modules defined.")

Custom Tucker Factorized Modules defined.
time: 0.00s
cummulative_time: 55.85s


## Step 3: Structural Replacement Across All 26 Layers & 78 Projections

We run DBSCAN clustering and Adam GD Tucker (`[3, 100, 350]`) across all 26 layers, physically replace `gate_proj`, `up_proj`, and `down_proj`, delete dense references, and purge CUDA cache.

In [7]:
# =====================================================================
# STEP 7: Structural Module Replacement across All 26 Layers
# =====================================================================
def optimize_tucker_gd(T, ranks=[3, 100, 350], num_steps=35, lr=1e-3, device="cpu"):
    core_init, factors_init = tucker(T, rank=ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    return core_param.detach(), [f.detach() for f in factors_param]

CHUNK_SIZE = 400
NUM_CHUNKS = 6
TUCKER_RANKS = [3, 100, 350]
DBSCAN_EPS = 0.06
DBSCAN_MIN_SAMPLES = 40

print(f"Applying Structural Replacement Across all {NUM_LAYERS} layers (78 projections)...")

for l in range(NUM_LAYERS):
    acts_l = acts_matrices[l]
    coord_rep_values = np.mean(acts_l, axis=0)

    db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="euclidean")
    labels = db.fit_predict(coord_rep_values.reshape(-1, 1))

    max_mags = np.max(np.abs(acts_l), axis=0)
    variances = np.var(acts_l, axis=0)
    super_mask = (labels == -1) | (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    chunk_coords_list = []
    for lab in unique_labels:
        c_indices = np.where((labels == lab) & (~super_mask))[0]
        if len(c_indices) == 0:
            continue
        sorted_indices = c_indices[np.argsort(coord_rep_values[c_indices])]
        num_full = len(sorted_indices) // CHUNK_SIZE
        for ci in range(num_full):
            chunk_coords_list.append(sorted_indices[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(chunk_coords_list) >= NUM_CHUNKS:
                break
        if len(chunk_coords_list) >= NUM_CHUNKS:
            break

    if len(chunk_coords_list) < NUM_CHUNKS:
        all_assigned = set(np.concatenate(chunk_coords_list) if chunk_coords_list else [])
        avail = [i for i in range(acts_l.shape[1]) if i not in all_assigned and not super_mask[i]]
        needed = NUM_CHUNKS - len(chunk_coords_list)
        for ci in range(needed):
            if len(avail) >= CHUNK_SIZE:
                chunk_coords_list.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]

    active_coords = np.concatenate(chunk_coords_list)

    layer_mod = model.model.layers[l]
    W_gate = layer_mod.mlp.gate_proj.weight.data.cpu()
    W_up = layer_mod.mlp.up_proj.weight.data.cpu()
    W_down = layer_mod.mlp.down_proj.weight.data.cpu()

    # 1. gate_proj
    T_gate = torch.stack([W_gate[c, :].float() for c in chunk_coords_list], dim=0)
    cg, fg = optimize_tucker_gd(T_gate, ranks=TUCKER_RANKS, device="cpu")
    fact_gate = TuckerFactorizedRowLinear(W_gate, super_indices, active_coords, cg, fg).to(model.device)
    del layer_mod.mlp.gate_proj
    layer_mod.mlp.gate_proj = fact_gate

    # 2. up_proj
    T_up = torch.stack([W_up[c, :].float() for c in chunk_coords_list], dim=0)
    cu, fu = optimize_tucker_gd(T_up, ranks=TUCKER_RANKS, device="cpu")
    fact_up = TuckerFactorizedRowLinear(W_up, super_indices, active_coords, cu, fu).to(model.device)
    del layer_mod.mlp.up_proj
    layer_mod.mlp.up_proj = fact_up

    # 3. down_proj
    T_down = torch.stack([W_down[:, c].T.float() for c in chunk_coords_list], dim=0)
    cd, fd = optimize_tucker_gd(T_down, ranks=TUCKER_RANKS, device="cpu")
    fact_down = TuckerFactorizedColLinear(W_down, super_indices, active_coords, cd, fd).to(model.device)
    del layer_mod.mlp.down_proj
    layer_mod.mlp.down_proj = fact_down

    del W_gate, W_up, W_down, T_gate, T_up, T_down
    clear_cuda_cache()

factorized_param_vram = get_model_param_vram_mb(model)
static_vram_saved = unprocessed_param_vram - factorized_param_vram

print(f"\nStructural Replacement Finished across all {NUM_LAYERS} layers.")
print(f"Pristine Parameter VRAM:      {unprocessed_param_vram:.2f} MB")
print(f"Factorized Parameter VRAM:    {factorized_param_vram:.2f} MB")
print(f"Net Static GPU Memory Saved:  {static_vram_saved:.2f} MB ({static_vram_saved/unprocessed_param_vram*100:.2f}% of full model parameters)")

Applying Structural Replacement Across all 26 layers (78 projections)...

Structural Replacement Finished across all 26 layers.
Pristine Parameter VRAM:      3814.26 MB
Factorized Parameter VRAM:    3158.84 MB
Net Static GPU Memory Saved:  655.42 MB (17.18% of full model parameters)
time: 136.23s
cummulative_time: 192.09s


## Step 4: Factorized Model Inference & Peak Runtime VRAM Profiling

We evaluate the factorized model on GLUE MNLI while recording allocated memory and peak runtime memory.

In [8]:
# =====================================================================
# STEP 8: Factorized Model Inference & Peak Runtime VRAM Profiling
# =====================================================================
clear_cuda_cache()

fact_predictions = []
fact_ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Factorized Model Inference & Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        fact_predictions.append(pred_label)
        fact_ground_truth.append(sample["label"])

factorized_snapshot = get_cuda_memory_snapshot()
factorized_accuracy = accuracy_score(fact_ground_truth, fact_predictions)
factorized_peak_vram = factorized_snapshot["max_allocated_mb"]
accuracy_delta = factorized_accuracy - baseline_accuracy

peak_vram_saved = baseline_peak_vram - factorized_peak_vram
allocated_vram_saved = baseline_snapshot["allocated_mb"] - factorized_snapshot["allocated_mb"]

print(f"\nFactorized Model Profile:")
print(f"  Downstream Accuracy:     {factorized_accuracy * 100:.2f}% (Δ: {accuracy_delta * 100:+.2f}%)")
print(f"  Static Parameter VRAM:   {factorized_param_vram:.2f} MB (Saved: {static_vram_saved:.2f} MB)")
print(f"  Allocated GPU VRAM:      {factorized_snapshot['allocated_mb']:.2f} MB (Saved: {allocated_vram_saved:.2f} MB)")
print(f"  Peak Runtime VRAM:       {factorized_peak_vram:.2f} MB (Saved: {peak_vram_saved:.2f} MB)")

Factorized Model Inference & Profiling: 100%|██████████| 1000/1000 [00:58<00:00, 17.13it/s]


Factorized Model Profile:
  Downstream Accuracy:     34.40% (Δ: -13.60%)
  Static Parameter VRAM:   3158.84 MB (Saved: 655.42 MB)
  Allocated GPU VRAM:      3291.23 MB (Saved: 651.21 MB)
  Peak Runtime VRAM:       3477.61 MB (Saved: 651.20 MB)
time: 58.66s
cummulative_time: 250.75s


## Step 5: Comparative VRAM Synthesis & Artifact Export

We generate the comparative VRAM table and export physical memory metrics to `artifacts/02_vram_all_mlp_dbscan_results.json`.

In [9]:
# =====================================================================
# STEP 9: Summary Table & JSON Artifact Export
# =====================================================================
print("=" * 105)
print(f"{'VRAM Metric':<32} | {'Unprocessed Baseline':<22} | {'All-MLP DBSCAN Tucker':<22} | {'Reduction / Delta':<20}")
print("=" * 105)
print(f"{'Static Parameter VRAM':<32} | {unprocessed_param_vram:>18.2f} MB | {factorized_param_vram:>18.2f} MB | {-static_vram_saved:>12.2f} MB ({-static_vram_saved/unprocessed_param_vram*100:.1f}%)")
print(f"{'Allocated GPU Memory':<32} | {baseline_snapshot['allocated_mb']:>18.2f} MB | {factorized_snapshot['allocated_mb']:>18.2f} MB | {-allocated_vram_saved:>12.2f} MB ({-allocated_vram_saved/baseline_snapshot['allocated_mb']*100:.1f}%)")
print(f"{'Peak Runtime VRAM':<32} | {baseline_peak_vram:>18.2f} MB | {factorized_peak_vram:>18.2f} MB | {-peak_vram_saved:>12.2f} MB ({-peak_vram_saved/baseline_peak_vram*100:.1f}%)")
print(f"{'GLUE MNLI Accuracy':<32} | {baseline_accuracy * 100:>18.2f} % | {factorized_accuracy * 100:>18.2f} % | {accuracy_delta * 100:>+12.2f} %")
print("=" * 105)

os.makedirs("artifacts", exist_ok=True)
vram_results = {
    "experiment": "02_vram_profiling_all_mlp_dbscan",
    "target_model": model_id,
    "ranks": TUCKER_RANKS,
    "num_layers": NUM_LAYERS,
    "num_projections": NUM_LAYERS * 3,
    "unprocessed_param_vram_mb": round(unprocessed_param_vram, 2),
    "factorized_param_vram_mb": round(factorized_param_vram, 2),
    "static_vram_saved_mb": round(static_vram_saved, 2),
    "static_vram_reduction_pct": round(static_vram_saved / unprocessed_param_vram * 100, 2),
    "baseline_peak_vram_mb": round(baseline_peak_vram, 2),
    "factorized_peak_vram_mb": round(factorized_peak_vram, 2),
    "peak_vram_saved_mb": round(peak_vram_saved, 2),
    "peak_vram_reduction_pct": round(peak_vram_saved / baseline_peak_vram * 100, 2),
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "factorized_accuracy": round(factorized_accuracy * 100, 2),
    "accuracy_delta": round(accuracy_delta * 100, 2),
    "timings": NOTEBOOK_TIMINGS,
}

with open("artifacts/02_vram_all_mlp_dbscan_results.json", "w") as f:
    json.dump(vram_results, f, indent=2)

print(f"Saved VRAM profiling results to artifacts/02_vram_all_mlp_dbscan_results.json")

VRAM Metric                      | Unprocessed Baseline   | All-MLP DBSCAN Tucker  | Reduction / Delta   
Static Parameter VRAM            |            3814.26 MB |            3158.84 MB |      -655.42 MB (-17.2%)
Allocated GPU Memory             |            3942.44 MB |            3291.23 MB |      -651.21 MB (-16.5%)
Peak Runtime VRAM                |            4128.81 MB |            3477.61 MB |      -651.20 MB (-15.8%)
GLUE MNLI Accuracy               |              48.00 % |              34.40 % |       -13.60 %
Saved VRAM profiling results to artifacts/02_vram_all_mlp_dbscan_results.json
time: 0.00s
cummulative_time: 250.76s
